# 03 Gold: baseline recommendation tables

Run 01_bronze.ipynb and 02_silver.ipynb first. This notebook builds API-ready Gold tables from Silver only. It never writes to Silver or Bronze.

## 1. Settings and source

Set FULL_REFRESH to True to rebuild Gold deliberately. Normal runs compare the Silver load log with Gold's source log and skip work when nothing changed.

In [1]:
FULL_REFRESH = False

In [2]:
from pathlib import Path
import time
import duckdb

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "crossselliq.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}. Run 01_bronze.ipynb first.")

con = duckdb.connect(str(DB_PATH))

def table_exists(schema, table):
    return con.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_schema = $schema AND table_name = $table",
        {"schema": schema, "table": table},
    ).fetchone()[0] == 1

required = ("customers", "articles", "transactions", "load_log")
missing = [name for name in required if not table_exists("silver", name)]
if missing:
    con.close()
    raise RuntimeError(f"Missing Silver tables: {missing}. Run 02_silver.ipynb first.")

print(f"Database: {DB_PATH}")


Database: c:\Users\diazv\Music\crosssell-iq\data\crossselliq.duckdb


## 2. Gold grain and rules

- gold.customers: one row per real customer, including customers without purchases. The API can distinguish them from unknown IDs.
- gold.customer_categories: one row per customer and recommendable product group. It records the first purchase date and number of transaction rows. Identical transaction rows remain separate purchases.
- gold.category_popularity: one row per recommendable product group. unique_buyers is the number of distinct buyers. popularity_score is unique_buyers divided by the largest category's unique_buyers, so the highest score is 1. It is not a purchase probability.
- gold.example_customers: up to three deterministic examples each with 0, 1, 2-3, and 4+ bought recommendable categories.

Unknown and empty product groups are excluded from Gold recommendations. Small named groups remain available. Customer and article attributes in Silver are current snapshots, so older transaction categories can change if article snapshots change.

A customer without rows in gold.customer_categories has no purchases in a recommendable category: either no purchases at all, or only Unknown articles (506 customers). Both get the same recommendations, so Gold does not separate them. The API must therefore not claim that such a customer never bought anything.

In [3]:
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

DDL = {
    "customers": """
        CREATE TABLE IF NOT EXISTS gold.customers (
            customer_id VARCHAR PRIMARY KEY
        )""",
    "customer_categories": """
        CREATE TABLE IF NOT EXISTS gold.customer_categories (
            customer_id VARCHAR NOT NULL,
            category VARCHAR NOT NULL,
            first_purchase_date DATE NOT NULL,
            purchase_count BIGINT NOT NULL,
            PRIMARY KEY (customer_id, category)
        )""",
    "category_popularity": """
        CREATE TABLE IF NOT EXISTS gold.category_popularity (
            category VARCHAR PRIMARY KEY,
            unique_buyers BIGINT NOT NULL,
            popularity_score DOUBLE NOT NULL
        )""",
    "example_customers": """
        CREATE TABLE IF NOT EXISTS gold.example_customers (
            customer_id VARCHAR PRIMARY KEY,
            purchased_category_count BIGINT NOT NULL
        )""",
    "source_log": """
        CREATE TABLE IF NOT EXISTS gold.source_log (
            file_name VARCHAR PRIMARY KEY,
            table_name VARCHAR NOT NULL,
            silver_processed_at TIMESTAMP NOT NULL,
            row_count BIGINT NOT NULL
        )""",
}
for ddl in DDL.values():
    con.execute(ddl)

source_difference = con.execute("""
    SELECT count(*) FROM (
        (SELECT file_name, table_name, processed_at, row_count FROM silver.load_log
         EXCEPT
         SELECT file_name, table_name, silver_processed_at, row_count FROM gold.source_log)
        UNION ALL
        (SELECT file_name, table_name, silver_processed_at, row_count FROM gold.source_log
         EXCEPT
         SELECT file_name, table_name, processed_at, row_count FROM silver.load_log)
    )
""").fetchone()[0]
gold_empty = any(con.execute(f"SELECT count(*) FROM gold.{name}").fetchone()[0] == 0
                 for name in ("customers", "customer_categories", "category_popularity", "example_customers"))
refresh_needed = FULL_REFRESH or source_difference > 0 or gold_empty
print(f"Silver/Gold source-log differences: {source_difference}")
print(f"Gold refresh needed: {refresh_needed}")


Silver/Gold source-log differences: 0
Gold refresh needed: False


## 3. Build and merge the baseline tables

Because Silver transactions do not retain their source filename, a changed Silver file triggers a complete Gold recomputation. The refresh and source-log update share one transaction. A run with unchanged Silver input skips this work.

In [4]:
if refresh_needed:
    started = time.perf_counter()
    con.begin()
    try:
        con.execute("""
            CREATE OR REPLACE TEMP TABLE stage_customers AS
            SELECT customer_id FROM silver.customers
        """)
        con.execute("""
            CREATE OR REPLACE TEMP TABLE stage_customer_categories AS
            SELECT
                t.customer_id,
                a.product_group_name AS category,
                min(t.transaction_date) AS first_purchase_date,
                count(*) AS purchase_count
            FROM silver.transactions t
            JOIN silver.articles a USING (article_id)
            WHERE a.product_group_name IS NOT NULL
              AND trim(a.product_group_name) <> ''
              AND a.product_group_name <> 'Unknown'
            GROUP BY t.customer_id, a.product_group_name
        """)
        con.execute("""
            CREATE OR REPLACE TEMP TABLE stage_category_popularity AS
            WITH categories AS (
                SELECT DISTINCT product_group_name AS category
                FROM silver.articles
                WHERE product_group_name IS NOT NULL
                  AND trim(product_group_name) <> ''
                  AND product_group_name <> 'Unknown'
            ),
            buyers AS (
                SELECT category, count(*) AS unique_buyers
                FROM stage_customer_categories
                GROUP BY category
            ),
            counts AS (
                SELECT c.category, coalesce(b.unique_buyers, 0) AS unique_buyers
                FROM categories c
                LEFT JOIN buyers b USING (category)
            )
            SELECT
                category,
                unique_buyers,
                CASE WHEN max(unique_buyers) OVER () = 0 THEN 0.0
                     ELSE unique_buyers::DOUBLE / max(unique_buyers) OVER ()
                END AS popularity_score
            FROM counts
        """)
        con.execute("""
            CREATE OR REPLACE TEMP TABLE stage_example_customers AS
            WITH counts AS (
                SELECT c.customer_id, coalesce(b.purchased_category_count, 0) AS purchased_category_count
                FROM stage_customers c
                LEFT JOIN (
                    SELECT customer_id, count(*) AS purchased_category_count
                    FROM stage_customer_categories
                    GROUP BY customer_id
                ) b USING (customer_id)
            )
            SELECT customer_id, purchased_category_count
            FROM counts
            QUALIFY row_number() OVER (
                PARTITION BY CASE
                    WHEN purchased_category_count = 0 THEN 0
                    WHEN purchased_category_count = 1 THEN 1
                    WHEN purchased_category_count <= 3 THEN 2
                    ELSE 3
                END
                ORDER BY customer_id
            ) <= 3
        """)
        for name in ("example_customers", "customer_categories", "category_popularity", "customers"):
            con.execute(f"DELETE FROM gold.{name}")
        for name, key in (
            ("customers", "target.customer_id = source.customer_id"),
            ("customer_categories", "target.customer_id = source.customer_id AND target.category = source.category"),
            ("category_popularity", "target.category = source.category"),
            ("example_customers", "target.customer_id = source.customer_id"),
        ):
            con.execute(f"""
                MERGE INTO gold.{name} AS target
                USING stage_{name} AS source
                ON {key}
                WHEN MATCHED THEN UPDATE BY NAME
                WHEN NOT MATCHED THEN INSERT BY NAME
            """)
        con.execute("DELETE FROM gold.source_log")
        con.execute("""
            INSERT INTO gold.source_log (file_name, table_name, silver_processed_at, row_count)
            SELECT file_name, table_name, processed_at, row_count
            FROM silver.load_log
        """)
        con.commit()
    except Exception:
        con.rollback()
        raise
    print(f"Gold refreshed in {time.perf_counter() - started:.1f} s")
else:
    print("No Silver changes: Gold is already up to date.")


No Silver changes: Gold is already up to date.


## 4. Validate Gold against Silver

The checks fail the notebook if a key, relationship, category rule, score, or source count is wrong.

In [5]:
results = []

def check(name, passed):
    results.append((name, passed))
    print(f"  {'OK  ' if passed else 'FAIL'} {name}")

def scalar(sql):
    return con.execute(sql).fetchone()[0]

check("all Silver customers are in Gold",
      scalar("SELECT count(*) FROM gold.customers") == scalar("SELECT count(*) FROM silver.customers"))
check("Gold customer IDs are valid hashes",
      scalar("SELECT count(*) FROM gold.customers WHERE NOT regexp_full_match(customer_id, '[0-9a-f]{64}')") == 0)
check("all customer-category buyers are known",
      scalar("SELECT count(*) FROM gold.customer_categories ANTI JOIN gold.customers USING (customer_id)") == 0)
check("all customer-category values are known",
      scalar("SELECT count(*) FROM gold.customer_categories ANTI JOIN gold.category_popularity USING (category)") == 0)
check("no unknown or empty categories",
      scalar("""SELECT count(*) FROM gold.category_popularity
                WHERE category IS NULL OR trim(category) = '' OR category = 'Unknown'""") == 0)
check("positive purchase counts and valid first dates",
      scalar("""SELECT count(*) FROM gold.customer_categories
                WHERE purchase_count <= 0 OR first_purchase_date IS NULL""") == 0)
check("popularity buyer counts match customer-category rows",
      scalar("""SELECT count(*) FROM gold.category_popularity p
                LEFT JOIN (
                    SELECT category, count(*) AS buyers
                    FROM gold.customer_categories GROUP BY category
                ) b USING (category)
                WHERE p.unique_buyers <> coalesce(b.buyers, 0)""") == 0)
check("popularity scores are normalized",
      scalar("""SELECT count(*) FROM gold.category_popularity
                WHERE popularity_score < 0 OR popularity_score > 1 OR NOT isfinite(popularity_score)""") == 0
      and scalar("SELECT max(popularity_score) FROM gold.category_popularity") == 1.0)
check("example customers are real and have correct counts",
      scalar("""SELECT count(*) FROM gold.example_customers e
                LEFT JOIN gold.customers c USING (customer_id)
                LEFT JOIN (
                    SELECT customer_id, count(*) AS category_count
                    FROM gold.customer_categories GROUP BY customer_id
                ) b USING (customer_id)
                WHERE c.customer_id IS NULL
                   OR e.purchased_category_count <> coalesce(b.category_count, 0)""") == 0)
check("Gold source log matches Silver",
      scalar("""SELECT count(*) FROM (
                  (SELECT file_name, table_name, processed_at, row_count FROM silver.load_log
                   EXCEPT SELECT file_name, table_name, silver_processed_at, row_count FROM gold.source_log)
                  UNION ALL
                  (SELECT file_name, table_name, silver_processed_at, row_count FROM gold.source_log
                   EXCEPT SELECT file_name, table_name, processed_at, row_count FROM silver.load_log)
                )""") == 0)
failed = [name for name, passed in results if not passed]
if failed:
    raise AssertionError(f"{len(failed)} Gold check(s) failed: {failed}")
print(f"All {len(results)} Gold checks passed.")


  OK   all Silver customers are in Gold
  OK   Gold customer IDs are valid hashes
  OK   all customer-category buyers are known
  OK   all customer-category values are known
  OK   no unknown or empty categories
  OK   positive purchase counts and valid first dates
  OK   popularity buyer counts match customer-category rows
  OK   popularity scores are normalized
  OK   example customers are real and have correct counts
  OK   Gold source log matches Silver
All 10 Gold checks passed.


## 5. Inspect a small result

These summaries show the API-facing tables without loading full transaction data into Python.

In [6]:
for name in ("customers", "customer_categories", "category_popularity", "example_customers"):
    print(f"gold.{name}: {scalar(f'SELECT count(*) FROM gold.{name}'):,} rows")

print("Top categories by distinct buyers")
con.sql("""
    SELECT category, unique_buyers, round(popularity_score, 4) AS popularity_score
    FROM gold.category_popularity
    ORDER BY popularity_score DESC, category
    LIMIT 10
""").show()

print("Example customer history sizes")
con.sql("""
    SELECT purchased_category_count, count(*) AS customers
    FROM gold.example_customers
    GROUP BY purchased_category_count
    ORDER BY purchased_category_count
    LIMIT 12
""").show()


gold.customers: 1,371,980 rows
gold.customer_categories: 4,819,805 rows
gold.category_popularity: 18 rows
gold.example_customers: 12 rows
Top categories by distinct buyers
┌────────────────────┬───────────────┬──────────────────┐
│      category      │ unique_buyers │ popularity_score │
│      varchar       │     int64     │      double      │
├────────────────────┼───────────────┼──────────────────┤
│ Garment Upper body │       1092325 │              1.0 │
│ Garment Lower body │        927480 │           0.8491 │
│ Garment Full body  │        644925 │           0.5904 │
│ Underwear          │        517715 │            0.474 │
│ Accessories        │        491251 │           0.4497 │
│ Swimwear           │        426146 │           0.3901 │
│ Shoes              │        269195 │           0.2464 │
│ Socks & Tights     │        267096 │           0.2445 │
│ Nightwear          │        169597 │           0.1553 │
│ Bags               │          6742 │           0.0062 │
└───────────────

## 6. Close the database

Close the writer connection before running the API or another notebook.

In [7]:
con.close()